# Table 2 — Assignment Data
## Purpose
Captures every assignment event that occurs on a claim — from initial intake
through all reassignments, escalations, and referrals.

## Structure
One row per event per claim. Each claim has between 2 and 8 events depending
on its complexity (`Low` → 2–3 events, `Medium` → 3–5, `High` → 5–8).

## Key design decisions
- **Four event types:** `Assigned` (always first), `Reassigned`, `Escalated`, `Referred`.
  Weights: 60% Reassigned, 25% Escalated, 15% Referred for events 2+.
- **Activity Notes** — a narrative sentence generated from templates. This is the
  field the parsing step reads to extract the adjuster name and reason.
- **Assigned To Adjuster** and **Reason for Reassignment** are left blank intentionally.
  They are populated by parsing the Activity Notes text.
- **Trigger** — `System` (auto-route engine) or `Manual` (supervisor action).
  Initial assignments are 75% System. Reassignments are 55% Manual.
- **Timestamps** — start at FNOL time, advance 0–5 days + 1–10 hours per event.
- `random.seed(42)` is set in the Table 1 notebook; this notebook loads Table 1
  from the CSV so the seed state continues correctly.

In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta

# Load Table 1 — needed to match claim numbers, exposure IDs, and complexity
df_claims = pd.read_csv("table1_claim_exposure_data.csv").fillna("")
print(f"Loaded Table 1: {len(df_claims)} rows, {df_claims['Claim Number'].nunique()} claims")
df_claims.head(3)

Loaded Table 1: 31 rows, 20 claims


,Claim Number,Exposure ID,Line of Business,Claim Type,Jurisdiction/State,FNOL Date/Date Reported,Claim Create Date,Claim Close Date,Reopen Date,Coverage Type,Loss Cause,Litigation Flag,CAT Flag,Subrogation Flag,Complexity
0,CLM-00001,CLM-00001-EXP01,Auto,Collision,MI,05/20/2024,05/20/2024,07/05/2024,,Property,Collision - Fixed Object,N,Y,Y,Low
1,CLM-00001,CLM-00001-EXP02,Auto,Collision,MI,05/20/2024,05/20/2024,07/05/2024,,BI,Theft - Vehicle,N,N,N,Low
2,CLM-00002,CLM-00002-EXP01,General Liability,Products Liability,IL,08/17/2024,08/17/2024,,,Collision,Equipment Malfunction,N,N,N,Medium


## Adjuster roster
The 15 adjusters here must match Table 3 exactly (same names, same groups).
Each tuple is `(name, group, specialty)` — specialty is informational only.

In [2]:
adjusters = [
    ("Sarah Mitchell",    "West Zone Operations Ohio Team 6",         "Auto"),
    ("James Kowalski",    "West Zone Operations Illinois Team 4",     "Auto"),
    ("Linda Reyes",       "Rochester Liability Team 2",               "Liability"),
    ("Marcus Thompson",   "Columbus Property Team 3",                 "Property"),
    ("Patricia Chen",     "Cleveland Workers Comp Team 1",            "Workers Comp"),
    ("David O'Brien",     "ISS Special Investigations Unit",          "ISS/SIU"),
    ("Anita Patel",       "West Zone Operations Ohio Team 3",         "Auto"),
    ("Robert Harrington", "Pittsburgh Property Team 2",               "Property"),
    ("Kevin Walsh",       "CAT Response Team - Central",              "CAT/Auto"),
    ("Jennifer Nguyen",   "Rochester Liability Team 4",               "Liability"),
    ("Thomas Burke",      "Parkersburg MD Team 4",                    "Material Damage"),
    ("Maria Gonzalez",    "Cleveland Workers Comp Team 3",            "Workers Comp"),
    ("Steven Park",       "Parkersburg MD Team 2",                    "Material Damage"),
    ("Rachel Coleman",    "Senior Adjusters Unit",                    "All Lines"),
    ("Christopher Adams", "Subrogation Recovery Unit",                "Subrogation"),
]

## Activity Notes templates
Three to four sentence variants per event type.
The adjuster name and group are embedded so the parsing step can extract them.
A context phrase (action or flag) is appended to make each note realistic.

In [3]:
def make_notes(event_type, adj_name, group, context):
    """Build a realistic activity note for one assignment event."""
    if event_type == "Assigned":
        t = random.choice([
            f"Claim created and assigned to {adj_name} in group {group}. {context}",
            f"Initial FNOL intake complete. Claim routed to {adj_name} ({group}) based on loss location and LOB eligibility. {context}",
            f"New claim received. Assignment Engine routed to {adj_name} in {group}. {context}",
        ])
    elif event_type == "Reassigned":
        t = random.choice([
            f"Assigned to {adj_name} in {group}. {context}",
            f"Claim transferred to {adj_name} ({group}). {context}",
            f"Reassigned from prior adjuster to {adj_name} in {group}. {context}",
            f"Workload rebalance triggered. Claim moved to {adj_name} in {group}. {context}",
        ])
    elif event_type == "Escalated":
        t = random.choice([
            f"Claim escalated to {adj_name} in {group}. {context}",
            f"Complexity threshold exceeded. Escalated to senior handler {adj_name} in {group}. {context}",
            f"Supervisor review required. Escalation issued to {adj_name} ({group}). {context}",
        ])
    else:  # Referred
        t = random.choice([
            f"Manually referred to {adj_name} in {group}. {context}",
            f"Specialist referral issued to {adj_name} ({group}) for further review. {context}",
            f"Coverage question identified. Claim referred to {adj_name} in {group}. {context}",
        ])
    return t

# Pool of context phrases appended to each note
action_contexts = [
    "Action: New exposure added; policy coverage verified.",
    "Action: Loss cause updated from initial FNOL report.",
    "Flag: Activity Overdue - initial contact with insured not completed.",
    "Action: Supplement inspection requested by field appraiser.",
    "Action: Date of loss corrected per police report.",
    "Action: Exposure closed; partial payment issued.",
    "Flag: Litigation hold applied; outside counsel notified.",
    "Action: CAT team engaged due to widespread weather event in region.",
    "Action: Potential ISS involvement identified during recorded statement.",
    "Action: Subrogation opportunity flagged against third-party carrier.",
    "Action: Medical records requested; treating physician identified.",
    "Action: Independent Medical Examination (IME) ordered.",
    "Flag: Coverage dispute - policy exclusion under review by coverage counsel.",
    "Action: Insured requested status update; call completed and documented.",
    "Action: Assigned back to material damage specialist for reinspection.",
    "Action: Final payment authorized; all exposures closed.",
    "Action: Claimant's attorney contacted; representation letter received.",
    "Action: Rental authorization approved; vehicle in repair shop.",
    "Action: Total loss determination initiated; market value assessed.",
    "Flag: Workload threshold exceeded on prior adjuster - system-triggered transfer.",
    "Action: Recorded statement completed; liability assessment in progress.",
    "Flag: Adjuster OOO - claim reassigned to maintain contact SLA.",
]

# Reason codes used for non-initial events (informational — not put in the CSV directly,
# but they inform what context phrases are selected)
reason_codes = [
    "Initial Assignment",
    "Workload Rebalancing",
    "Adjuster Out of Office / PTO",
    "LOB Mismatch - Specialty Required",
    "Geographic Routing Correction",
    "Complexity Upgrade",
    "Litigation Referral",
    "SIU / ISS Escalation",
    "Coverage Review Required",
    "CAT Activation",
    "Subrogation Referral",
    "Activity Overdue - Supervisor Reassignment",
    "Functional Handoff - MD to Liability",
    "Return After Specialist Review",
]

## Trigger and assignment logic
- **Trigger** — whether the event was system-initiated or manual.
- **Assigned By** — auto-route engine for system triggers, a named supervisor for manual.

In [4]:
def get_trigger(event_type):
    """
    Assigned:   75% System (auto-route on FNOL), 25% Manual
    Reassigned: 45% System (workload/rule-based), 55% Manual (supervisor action)
    Escalated / Referred: always Manual
    """
    if event_type == "Assigned":
        return random.choices(["System", "Manual"], weights=[0.75, 0.25])[0]
    if event_type == "Reassigned":
        return random.choices(["System", "Manual"], weights=[0.45, 0.55])[0]
    return "Manual"

def get_assigned_by(trigger):
    if trigger == "System":
        return "Assignment Engine (Auto-Route)"
    return random.choice(["Rachel Coleman", "Supervisor - Team Lead", "Claims Manager"])

## Generate event rows
For each claim, generate events sequentially.
- Event 1 is always `Assigned`.
- Subsequent events pick a new adjuster (never the same as the current one)
  and a new event type with the weights above.
- Timestamps advance realistically from the FNOL datetime.

In [5]:
random.seed(42)  # match seed from Table 1 generation

claim_meta = df_claims.drop_duplicates("Claim Number").set_index("Claim Number")
assignment_rows = []

for claim_num in df_claims["Claim Number"].unique():
    meta       = claim_meta.loc[claim_num]
    complexity = meta["Complexity"]
    fnol_dt    = datetime.strptime(meta["FNOL Date/Date Reported"], "%m/%d/%Y")
    exposures  = df_claims[df_claims["Claim Number"] == claim_num]["Exposure ID"].tolist()

    # Number of events scales with complexity
    num_events = {
        "Low":    random.randint(2, 3),
        "Medium": random.randint(3, 5),
        "High":   random.randint(5, 8),
    }[complexity]

    current_time = fnol_dt + timedelta(hours=random.randint(0, 4))
    first_adj    = random.choice(adjusters)
    adj_name, adj_group = first_adj[0], first_adj[1]

    for ev_idx in range(1, num_events + 1):
        exp_id = random.choice(exposures)

        if ev_idx == 1:
            event_type = "Assigned"
        else:
            event_type = random.choices(
                ["Reassigned", "Escalated", "Referred"],
                weights=[0.60, 0.25, 0.15],
            )[0]
            new_adj = random.choice([a for a in adjusters if a[0] != adj_name])
            adj_name, adj_group = new_adj[0], new_adj[1]

        context = random.choice(action_contexts)
        trigger = get_trigger(event_type)

        assignment_rows.append({
            "Claim Number":           claim_num,
            "Exposure ID":            exp_id,
            "Assignment #":           ev_idx,
            "Event Type":             event_type,
            "Timestamp":              current_time.strftime("%m/%d/%Y %I:%M %p"),
            "Activity Notes":         make_notes(event_type, adj_name, adj_group, context),
            "Assigned To Adjuster":   "",   # blank — populated by parsing Activity Notes
            "Reason for Reassignment": "",  # blank — populated by parsing Activity Notes
            "Manual/System Trigger":  trigger,
            "Assigned By":            get_assigned_by(trigger),
            "Complexity":             complexity,
        })

        current_time += timedelta(days=random.randint(0, 5), hours=random.randint(1, 10))

df_assignments = pd.DataFrame(assignment_rows)
df_assignments.to_csv("table2_assignment_data.csv", index=False)
print(f"Table 2: {len(df_assignments)} rows saved to table2_assignment_data.csv")
df_assignments.head(12)

Table 2: 60 rows saved to table2_assignment_data.csv


,Claim Number,Exposure ID,Assignment #,Event Type,Timestamp,Activity Notes,Assigned To Adjuster,Reason for Reassignment,Manual/System Trigger,Assigned By,Complexity
0,CLM-00001,CLM-00001-EXP01,1,Assigned,05/20/2024 01:00 AM,New claim received. Assignment Engine routed t...,,,System,Assignment Engine (Auto-Route),Low
1,CLM-00001,CLM-00001-EXP02,2,Reassigned,05/20/2024 11:00 AM,Assigned to James Kowalski in West Zone Operat...,,,System,Assignment Engine (Auto-Route),Low
2,CLM-00002,CLM-00002-EXP01,1,Assigned,08/17/2024 04:00 AM,Initial FNOL intake complete. Claim routed to ...,,,System,Assignment Engine (Auto-Route),Medium
3,CLM-00002,CLM-00002-EXP01,2,Referred,08/19/2024 07:00 AM,Manually referred to Anita Patel in West Zone ...,,,Manual,Supervisor - Team Lead,Medium
4,CLM-00002,CLM-00002-EXP01,3,Escalated,08/19/2024 01:00 PM,Supervisor review required. Escalation issued ...,,,Manual,Supervisor - Team Lead,Medium
5,CLM-00003,CLM-00003-EXP02,1,Assigned,06/21/2024 04:00 AM,Claim created and assigned to Christopher Adam...,,,System,Assignment Engine (Auto-Route),Low
6,CLM-00003,CLM-00003-EXP02,2,Referred,06/21/2024 08:00 AM,Manually referred to Rachel Coleman in Senior ...,,,Manual,Supervisor - Team Lead,Low
7,CLM-00003,CLM-00003-EXP02,3,Reassigned,06/23/2024 04:00 PM,Assigned to David O'Brien in ISS Special Inves...,,,Manual,Claims Manager,Low
8,CLM-00004,CLM-00004-EXP01,1,Assigned,02/05/2024 03:00 AM,Claim created and assigned to Patricia Chen in...,,,System,Assignment Engine (Auto-Route),Low
9,CLM-00004,CLM-00004-EXP01,2,Reassigned,02/06/2024 04:00 AM,Reassigned from prior adjuster to James Kowals...,,,Manual,Rachel Coleman,Low


## Validation — event type distribution

In [6]:
print("Event type counts:")
print(df_assignments["Event Type"].value_counts().to_string())
print()
print("Events per claim (sample):")
print(df_assignments.groupby("Claim Number").size().describe())

Event type counts:
Event Type
Assigned      20
Reassigned    18
Referred      11
Escalated     11

Events per claim (sample):
count    20.000000
mean      3.000000
std       0.725476
min       2.000000
25%       3.000000
50%       3.000000
75%       3.000000
max       5.000000
dtype: float64
